# Example 11.4 / 1: Domain-safe formulation

Companion notebook for **11.4 Pitfalls of Nonlinear Programming** in the AMPL lab Documentation, section "Domain-safe formulation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Domain-safe formulation

For a term $x/(1-x/L)$, the denominator vanishes at $x=L$. A valid model needs a physical limit below that singularity, not merely a convenient initial value.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param limit > 0;
param demand >= 0, < limit;

var Flow >= 0, <= 0.99 * limit, := 0.5 * limit;

minimize CongestionCost:
    Flow / (1 - Flow / limit);

subject to ServeDemand:
    Flow >= demand;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["limit"] = 10
ampl.param["demand"] = 6

In [5]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result)
print(round(ampl.var["Flow"].value(), 6))
print(round(ampl.con["ServeDemand"].slack(), 6))

solved
6.0
0.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param limit := 10;
param demand := 6;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver ipopt;
option ipopt_options 'outlev=0';
option solver_msg 0;
solve;
display Flow, ServeDemand.slack;

Ipopt 3.12.13: outlev=0


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************


suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
Flow = 6
ServeDemand.slack = 0

